# 01 · 첫 시세 조회

집에서 셋업 직후 가장 먼저 돌려볼 노트북.

**전제조건**
1. `.env` 파일이 채워져 있다 (`KIS_VIRTUAL_APPKEY` 등)
2. `pip install -r requirements.txt` 완료
3. 모의투자 계좌가 신청·승인되었다

**이 노트북이 검증하는 것**
- 환경변수 로드
- KIS 토큰 발급
- 삼성전자 현재가 조회
- 일봉 60일 조회 + 시각화
- 골든크로스 전략 신호 생성

## 0. 프로젝트 루트로 경로 잡기

In [ ]:
import sys
from pathlib import Path

# 노트북이 notebooks/ 에 있어도 src/ 를 임포트 가능하게
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

## 1. 환경변수 + 토큰 점검

In [ ]:
from src.config import settings
from src.kis_auth import get_token

print(f'MODE:      {settings.mode.value}')
print(f'Base URL:  {settings.base_url}')
print(f'계좌:      {settings.account_full}')

settings.validate_runtime()
token = get_token()
print(f'\n✅ 토큰 OK (만료: {token.expires_at:%Y-%m-%d %H:%M})')

## 2. 삼성전자 현재가 조회

In [ ]:
from src.kis_client import KISClient

client = KISClient()
resp = client.get_price('005930')  # 삼성전자
print(f"현재가: {int(resp['output']['stck_prpr']):,}원")
print(f"전일대비: {resp['output']['prdy_vrss']}원 ({resp['output']['prdy_ctrt']}%)")

## 3. 일봉 60일 조회 + 시각화

In [ ]:
from src.bot.runner import fetch_recent_history
import matplotlib.pyplot as plt

history = fetch_recent_history(client, '005930', days=60)
print(history.tail())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.index, history['close'], label='Close')
ax.plot(history.index, history['close'].rolling(5).mean(), label='MA5')
ax.plot(history.index, history['close'].rolling(20).mean(), label='MA20')
ax.set_title('Samsung Electronics (005930) — Last 60 days')
ax.legend()
plt.tight_layout()
plt.show()

## 4. 골든크로스 전략 신호 생성

In [ ]:
from src.strategies.golden_cross import GoldenCrossStrategy

strat = GoldenCrossStrategy(short_window=5, long_window=20)
signal = strat.generate_signal('005930', history)
print(f'신호: {signal.type.value}')
print(f'가격: {signal.price:,.0f}원')
print(f'이유: {signal.reason}')

## 다음 단계

- [ ] `python -m src.backtest.runner --strategy golden_cross --symbol 005930 --from 2023-01-01 --to 2024-12-31`
- [ ] `python -m src.bot.runner --strategy golden_cross --symbol 005930 --dry-run`
- [ ] 모의투자에서 1주일 운영 후 백테스트와 결과 괴리 분석
- [ ] 전략 추가·개선 (필터, 다종목 동시, 손절/익절 등)